In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.__version__)

2.8.0


In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from seq2seq.model.encoder import Encoder
from seq2seq.model.decoder import Decoder

In [4]:
SRC_VOCAB_SIZE = 100
TGT_VOCAB_SIZE = 100

BATCH_SIZE = 4

SRC_LENGTH = 7
TGT_LENGTH = 7

NUM_LAYERS = 2
EMBEDDING_DIM = 32
HIDDEN_DIM = 64

In [5]:
class Seq2Seq(nn.Module):
    
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        target_vocab_size: int,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.target_vocab_size = target_vocab_size

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        # full_teacher_forcing: bool,
    ):
        
        batch_size = src.shape[0]
        target_length = tgt.shape[1]

        _, hidden, cell = self.encoder(src)

        # Start with the first target token as the first input which is <SOS>
        input_token = tgt[:, 0]

        all_logits = torch.zeros(
            batch_size,
            tgt.shape[1] - 1,
            self.target_vocab_size,
            device=src.device
        )

        # First time step t=1, input token is still at index 0 for <SOS>
        for t in range(1, target_length):
            logits, hidden, cell = self.decoder(
                input_token,
                hidden,
                cell
            )
            
            all_logits[:, t - 1, :] = logits

            input_token = tgt[:, t]

            ''' Implemented generate for generation without full teacher forcing
            if full_teacher_forcing:
                input_token = tgt[:, t]
            else:
                input_token = logits.argmax(dim=-1)
            '''

        # Return all logits for each time position, for each sequence in the batch
        return all_logits

    def generate(
        self,
        src: torch.Tensor,
        sos_idx: int,
        max_length: int,
    ) -> torch.Tensor:

        batch_size = src.shape[0]

        _, hidden, cell = self.encoder(src)
        
        input_token = torch.full(
            (batch_size,),
            sos_idx,
            dtype=torch.long,
            device=src.device
        )

        generated_steps = []

        for _ in range(max_length):
            logits, hidden, cell = self.decoder(
                input_token,
                hidden,
                cell
            )

            input_token = logits.argmax(dim=-1)
            generated_steps.append(input_token)

        return torch.stack(generated_steps, dim=1)

In [6]:
encoder = Encoder(
    vocab_size=SRC_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
)

decoder = Decoder(
    vocab_size=TGT_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
)

src = torch.randint(
    low=0,
    high=SRC_VOCAB_SIZE,
    size=(BATCH_SIZE, SRC_LENGTH)
)

SOS_IDX = 1

tgt = torch.randint(
    low=0,
    high=TGT_VOCAB_SIZE,
    size=(BATCH_SIZE, TGT_LENGTH)
)

tgt[:, 0] = SOS_IDX

model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    target_vocab_size=TGT_VOCAB_SIZE,
)

all_logits = model(src, tgt, False)

In [52]:
all_logits.shape

torch.Size([4, 6, 100])

In [53]:
targets = tgt[:, 1:]
targets.shape

torch.Size([4, 6])

In [54]:
loss = F.cross_entropy(
    all_logits.reshape(-1, TGT_VOCAB_SIZE), # Flatten all of the logits for all of the batches
    targets.reshape(-1), # Flatten all of the batches true targets
)

loss.item()

4.621119976043701

In [55]:
loss.backward()

In [29]:
assert encoder.embedding.weight.grad is not None
assert encoder.lstm.weight_ih_l0.grad is not None
assert decoder.embedding.weight.grad is not None
assert decoder.lstm.weight_ih_l0.grad is not None
assert decoder.output.weight.grad is not None
assert torch.isfinite(loss)

In [30]:
assert encoder.embedding.weight.grad.abs().sum() > 0
assert decoder.output.weight.grad.abs().sum() > 0